# NIFTY Recovery Evidence Lab
## 01 — Data Acquisition

### Objective

Acquire historical daily NIFTY 50 data for investigating
post-event recovery patterns following significant
one-day declines.

### Data Source

- Provider: Yahoo Finance
- Instrument: NIFTY 50
- Ticker: ^NSEI
- Frequency: Daily
- Required fields: Date, Open, High, Low, Close

### Research Principles

- Preserve the original downloaded data.
- Validate date coverage and OHLC values.
- Document cleaning decisions.
- Avoid look-ahead bias.
- Ensure reproducibility.

In [1]:
import pandas as pd
import yfinance as yf
from pathlib import Path

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download historical NIFTY 50 daily data
ticker = "^NSEI"

df = yf.download(
    ticker,
    start="2007-01-01",
    end="2026-09-22",
    auto_adjust=False,
    progress=False,
    actions=False
)

df.head()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,,
2007-09-17,4494.649902,4494.649902,4549.049805,4482.850098,4518.450195,0
2007-09-18,4546.200195,4546.200195,4551.799805,4481.549805,4494.100098,0
2007-09-19,4732.350098,4732.350098,4739.000000,4550.250000,4550.250000,0
2007-09-20,4747.549805,4747.549805,4760.850098,4721.149902,4734.850098,0
2007-09-21,4837.549805,4837.549805,4855.700195,4733.700195,4752.950195,0


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

Shape: (4664, 6)

Columns:
MultiIndex([('Adj Close', '^NSEI'),
            (    'Close', '^NSEI'),
            (     'High', '^NSEI'),
            (      'Low', '^NSEI'),
            (     'Open', '^NSEI'),
            (   'Volume', '^NSEI')],
           names=['Price', 'Ticker'])

Data types:
Price      Ticker
Adj Close  ^NSEI     float64
Close      ^NSEI     float64
High       ^NSEI     float64
Low        ^NSEI     float64
Open       ^NSEI     float64
Volume     ^NSEI       int64
dtype: object

First 5 rows:


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,,
2007-09-17,4494.649902,4494.649902,4549.049805,4482.850098,4518.450195,0
2007-09-18,4546.200195,4546.200195,4551.799805,4481.549805,4494.100098,0
2007-09-19,4732.350098,4732.350098,4739.000000,4550.250000,4550.250000,0
2007-09-20,4747.549805,4747.549805,4760.850098,4721.149902,4734.850098,0
2007-09-21,4837.549805,4837.549805,4855.700195,4733.700195,4752.950195,0



Last 5 rows:


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,,
2026-09-15,23118.599609,23118.599609,23592.849609,23118.599609,23576.150391,338800
2026-09-16,23217.599609,23217.599609,23284.750000,23116.099609,23201.599609,261600
2026-09-17,23270.599609,23270.599609,23363.550781,23193.650391,23195.250000,259700
2026-09-18,23346.400391,23346.400391,23389.150391,23286.599609,23334.699219,375300
2026-09-21,23414.300781,23414.300781,23466.800781,23314.800781,23330.199219,213600


In [4]:
raw_file = RAW_DATA_DIR / "nifty50_yahoo_raw.csv"

df.to_csv(raw_file)

print(f"Raw data saved to: {raw_file}")

Raw data saved to: c:\Users\anvit\OneDrive\Desktop\Clients\nifty recovery evidence lab\data\raw\nifty50_yahoo_raw.csv


In [5]:
# Create a copy of the raw downloaded data
df_clean = df.copy()

# Flatten MultiIndex columns
if isinstance(df_clean.columns, pd.MultiIndex):
    df_clean.columns = df_clean.columns.get_level_values(0)

# Reset index to make Date a regular column
df_clean = df_clean.reset_index()

# Keep only the required research fields
required_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close"
]

df_clean = df_clean[required_columns]

# Ensure Date is in datetime format
df_clean["Date"] = pd.to_datetime(df_clean["Date"])

# Sort chronologically
df_clean = df_clean.sort_values("Date").reset_index(drop=True)

# Display the result
display(df_clean.head())

print("Shape:", df_clean.shape)
print("Columns:", df_clean.columns.tolist())

Price,Date,Open,High,Low,Close
0,2007-09-17,4518.450195,4549.049805,4482.850098,4494.649902
1,2007-09-18,4494.100098,4551.799805,4481.549805,4546.200195
2,2007-09-19,4550.250000,4739.000000,4550.250000,4732.350098
3,2007-09-20,4734.850098,4760.850098,4721.149902,4747.549805
4,2007-09-21,4752.950195,4855.700195,4733.700195,4837.549805


Shape: (4664, 5)
Columns: ['Date', 'Open', 'High', 'Low', 'Close']


In [6]:
# =====================================
# BASIC DATA VALIDATION
# =====================================

print("DATA VALIDATION REPORT")
print("=" * 40)

# 1. Date coverage
print("\n1. Date Coverage")
print("Start:", df_clean["Date"].min())
print("End:", df_clean["Date"].max())

# 2. Duplicate dates
duplicate_dates = df_clean["Date"].duplicated().sum()
print("\n2. Duplicate Dates:", duplicate_dates)

# 3. Missing values
print("\n3. Missing Values")
print(df_clean.isnull().sum())

# 4. Chronological ordering
is_sorted = df_clean["Date"].is_monotonic_increasing
print("\n4. Chronological Order:", is_sorted)

# 5. Invalid OHLC values
ohlc_columns = ["Open", "High", "Low", "Close"]

invalid_ohlc = (
    df_clean[ohlc_columns].isnull().any(axis=1)
    | (df_clean[ohlc_columns] <= 0).any(axis=1)
)

print("\n5. Invalid OHLC Rows:", invalid_ohlc.sum())

# 6. OHLC consistency
invalid_relationships = (
    (df_clean["High"] < df_clean[["Open", "Close"]].max(axis=1))
    | (df_clean["Low"] > df_clean[["Open", "Close"]].min(axis=1))
    | (df_clean["High"] < df_clean["Low"])
)

print("6. Invalid OHLC Relationships:", invalid_relationships.sum())

# 7. Final shape
print("\n7. Final Shape:", df_clean.shape)

DATA VALIDATION REPORT

1. Date Coverage
Start: 2007-09-17 00:00:00
End: 2026-09-21 00:00:00

2. Duplicate Dates: 0

3. Missing Values
Price
Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64

4. Chronological Order: True

5. Invalid OHLC Rows: 0
6. Invalid OHLC Relationships: 0

7. Final Shape: (4664, 5)


In [7]:
# Remove inherited column axis name
df_clean.columns.name = None

display(df_clean.head())

,Date,Open,High,Low,Close
0,2007-09-17,4518.450195,4549.049805,4482.850098,4494.649902
1,2007-09-18,4494.100098,4551.799805,4481.549805,4546.200195
2,2007-09-19,4550.250000,4739.000000,4550.250000,4732.350098
3,2007-09-20,4734.850098,4760.850098,4721.149902,4747.549805
4,2007-09-21,4752.950195,4855.700195,4733.700195,4837.549805


In [8]:
# =====================================
# SAVE PROCESSED DATASET
# =====================================

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

processed_file = PROCESSED_DATA_DIR / "nifty50_clean.csv"

df_clean.to_csv(processed_file, index=False)

print(f"Processed data saved to: {processed_file}")

Processed data saved to: c:\Users\anvit\OneDrive\Desktop\Clients\nifty recovery evidence lab\data\processed\nifty50_clean.csv


In [9]:
# Calculate daily close-to-close returns
df_clean["daily_return"] = df_clean["Close"].pct_change()

# Display the 10 largest absolute daily returns
extreme_returns = (
    df_clean
    .dropna(subset=["daily_return"])
    .assign(
        abs_return=lambda x: x["daily_return"].abs()
    )
    .sort_values("abs_return", ascending=False)
    .head(10)
)

display(
    extreme_returns[
        ["Date", "Close", "daily_return"]
    ]
)

,Date,Close,daily_return
402,2009-05-18,4323.149902,0.177441
3057,2020-03-23,7610.250000,-0.129805
274,2008-10-24,2584.000000,-0.122029
3066,2020-04-07,8792.200195,0.087632
86,2008-01-21,5208.799805,-0.087024
3050,2020-03-12,9590.150391,-0.083019
3052,2020-03-16,9197.400391,-0.076121
277,2008-10-31,2885.600098,0.069910
90,2008-01-25,5383.350098,0.069515
276,2008-10-29,2697.050049,0.068477
